# 04 · Teach NLLB that Ekegusii exists

**The problem.** NLLB-200 supports 202 languages. Ekegusii is not one of them —
there is no `guz_Latn` token, so there is no way to ask the model for Ekegusii
output. We have to add the language ourselves.

**Three things happen here**

1. Add a `guz_Latn` token to the tokenizer and grow the embedding matrix.
2. Initialise the new language embedding from **`kik_Latn` (Kikuyu)** rather than
   randomly. Kikuyu is a Kenyan Bantu language — a far better starting point than
   noise, and it makes early training much more stable.
3. Measure **subword fertility**: how many tokens the tokenizer needs per
   Ekegusii word. Ekegusii is agglutinative, so if fertility is much worse than
   Kiswahili's, sequences get long and training gets expensive.

**Outputs** — `artifacts/nllb600m-guz-init/`
**Runtime** — 2–3 minutes.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
C.use_house_style()
print(f"project root: {C.ROOT}")

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained(C.BASE_MODEL, src_lang=C.ENG, tgt_lang=C.SWH)
model = AutoModelForSeq2SeqLM.from_pretrained(C.BASE_MODEL)
print(f"vocab size      : {len(tok):,}")
# get_input_embeddings() rather than model.model.shared - it is the public
# API and does not depend on the internal module layout.
print(f"embedding shape : {tuple(model.get_input_embeddings().weight.shape)}")
print(f"parameters      : {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

## 1. Confirm Ekegusii really is absent

In [ ]:
# `tok.additional_special_tokens` was the documented way to do this, but newer
# transformers releases removed it from NllbTokenizer. C.nllb_language_tokens
# tries the documented attributes and falls back to scanning the vocabulary for
# the `xxx_Yyyy` pattern, so this works on any version.
lang_tokens = C.nllb_language_tokens(tok)

print(f"language tokens in NLLB-200: {len(lang_tokens)}")
print(f"guz_Latn present? {C.GUZ in lang_tokens}")
print(f"kik_Latn (Kikuyu) present? {'kik_Latn' in lang_tokens}")
print(f"kam_Latn (Kamba)  present? {'kam_Latn' in lang_tokens}")
print("\nKenyan / nearby Bantu languages NLLB does cover:")
print(" ", [t for t in lang_tokens if t.split('_')[0] in
      {'kik', 'kam', 'luo', 'swh', 'lug', 'nya', 'kin', 'run',
       'bem', 'sna', 'xho', 'zul', 'som'}])

## 2. Add the token and grow the embedding matrix

`resize_token_embeddings` appends randomly-initialised rows. We immediately
overwrite the new language row with Kikuyu's, so the model starts from
"something Bantu and Kenyan" instead of noise.

In [ ]:
before = len(tok)

# add_language_token() adds guz_Latn, resizes the embedding matrix, and copies
# the kik_Latn row into the new slot. It uses add_tokens(special_tokens=True)
# rather than add_special_tokens({"additional_special_tokens": ...}), which
# needs the attribute this transformers version no longer has - and which in
# some versions REPLACES the language list instead of extending it.
guz_id, src_id = C.add_language_token(tok, model, C.GUZ, C.GUZ_INIT_FROM)

print(f"vocab {before:,} -> {len(tok):,}")
print(f"{C.GUZ} id = {guz_id}   initialised from {C.GUZ_INIT_FROM} (id {src_id})")

emb = model.get_input_embeddings().weight
print(f"cosine(guz, kik) after init: "
      f"{torch.nn.functional.cosine_similarity(emb[guz_id], emb[src_id], dim=0):.4f}")
print(f"embedding matrix now {tuple(emb.shape)}")

## 3. Encoding helper — build the special tokens explicitly

NLLB's expected format is `[lang_code] … tokens … [eos]` on both the source and
the target side. Rather than relying on `tokenizer.src_lang` / `tgt_lang`
internals — whose behaviour has changed between `transformers` releases and
which do not know about our new language — we build the sequences ourselves.
It is three lines, it is version-proof, and you can see exactly what the model
receives.

In [ ]:
EOS = tok.eos_token_id

def encode(text, lang, max_len=128):
    ids = tok(text, add_special_tokens=False, truncation=True,
              max_length=max_len - 2)["input_ids"]
    return [tok.convert_tokens_to_ids(lang)] + ids + [EOS]

demo = "Report suspected cholera cases to the nearest health facility."
ids = encode(demo, C.ENG)
print("ids   :", ids[:12], "...")
print("tokens:", tok.convert_ids_to_tokens(ids)[:12], "...")
print("decode:", tok.decode(ids, skip_special_tokens=False)[:110])

guz_demo = encode("Manya ogotebia ekitengo k'obogima.", C.GUZ)
print("\nEkegusii labels start with the new language token:")
print(" ", tok.convert_ids_to_tokens(guz_demo)[:8])

## 4. Subword fertility — how expensive is Ekegusii?

Tokens per word. English sits near 1.2 for NLLB. If Ekegusii comes in far above
Kiswahili, sequences are longer, batches are smaller and training is slower —
and it tells you the tokenizer is fragmenting the morphology.

In [ ]:
bible = pd.read_csv(C.BIBLE_CSV).sample(2000, random_state=C.SEED)

def fertility(texts):
    tokens = words = 0
    for t in texts:
        t = str(t)
        words += len(t.split())
        tokens += len(tok(t, add_special_tokens=False)["input_ids"])
    return tokens / max(1, words)

rows = [("English", fertility(bible.english), C.LANG_COLOR["english"]),
        ("Kiswahili", fertility(bible.swahili), C.LANG_COLOR["swahili"]),
        ("Ekegusii", fertility(bible.ekegusii), C.LANG_COLOR["ekegusii"])]

fig, ax = plt.subplots(figsize=(8, 3.4))
bars = ax.barh([r[0] for r in rows][::-1], [r[1] for r in rows][::-1],
               color=[r[2] for r in rows][::-1], height=0.6)
for b, r in zip(bars, rows[::-1]):
    ax.annotate(f"{r[1]:.2f}", (r[1], b.get_y() + b.get_height()/2), xytext=(6, 0),
                textcoords="offset points", va="center", color=C.INK_MUTED)
ax.set_title("Subword fertility — NLLB tokens per word")
ax.set_xlabel("tokens / word"); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlim(0, max(r[1] for r in rows) * 1.18)
C.save_fig(fig, "04_subword_fertility"); plt.show()

f_en, f_sw, f_gz = [r[1] for r in rows]
print(f"Ekegusii costs {f_gz/f_en:.2f}x more tokens per word than English,")
print(f"and {f_gz/f_sw:.2f}x more than Kiswahili.")
print("Above ~2x versus Kiswahili, consider extending the SentencePiece vocabulary")
print("before training; below that, the stock vocabulary is workable.")

## 5. Save the extended model and tokenizer

In [ ]:
C.EXTENDED_MODEL.mkdir(parents=True, exist_ok=True)
model.save_pretrained(C.EXTENDED_MODEL)
tok.save_pretrained(C.EXTENDED_MODEL)
C.save_json({"guz_id": int(guz_id), "init_from": C.GUZ_INIT_FROM,
             "vocab_size": len(tok),
             "fertility": {"english": f_en, "swahili": f_sw, "ekegusii": f_gz}},
            C.DATA / "tokenizer_extension.json")
print(f"\nsaved -> {C.EXTENDED_MODEL}")
print("Next: 05_finetune.ipynb")